# 🛡️ PROJECT 15: TWEET SENTIMENT EXTRACTION
> **Contextual Sub-string Extraction via Sequence Tagging Architecture**

Welcome to the 15th milestone of our Industrial Machine Learning MLOps pipeline. In this project, we transition from simple text classification to **Information Extraction**.

## 🎯 The Mission
The objective is not just to classify a tweet as positive, negative, or neutral, but to **extract the exact phrase** (substring) that exemplifies that sentiment. 

To solve this, we will frame the extraction problem as a **Sequence Tagging (Token Classification)** task using a Deep **Bidirectional LSTM** architecture.

## ⚙️ The 10-Step Compliance Framework
1. **Understand Goal:** Token-level sequence classification (Extraction).
2. **EDA:** Target variable distributions and missing values.
3. **Feature Selection:** Isolate `text`, `sentiment`, and `selected_text`.
4. **Categorical to Numeric:** Encode sentiment categories.
5. **Data Cleansing:** Null dropping and whitespace stripping.
6. **Feature Engineering:** Word counts and extraction ratios.
7. **Encoding (One-Hot/Tokenization):** Tokenize tweets and generate binary masks for the target extraction.
8. **Train/Test Split:** Isolate validation data for strict evaluation.
9. **Train & Predict:** Deploy Bi-LSTM with TimeDistributed Dense layers.
10. **Evaluate:** Model accuracy metrics and visualization.

---
*Let's ignite the Extraction Engine.* 🚀

In [1]:
# ==============================================================================
# 🛡️ SYSTEM SETUP & LIBRARIES
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import re

# Visualizations (Neon/Dark Theme)
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'kaggle'
from IPython.display import display, HTML

# Machine Learning & Keras Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, TimeDistributed, SpatialDropout1D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

2026-03-16 17:42:31.717260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773682952.080669      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773682952.199682      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773682953.095350      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773682953.095408      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773682953.095462      17 computation_placer.cc:177] computation placer alr

In [2]:
# --- STEP 2: READ AND INSPECT DATA (EDA) ---
print("📊 STEP 2: DATA INGESTION & EXPLORATORY DATA ANALYSIS (EDA)")

DATA_PATH = '/kaggle/input/competitions/tweet-sentiment-extraction/train.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print("✅ Full Dataset Loaded Successfully.")
except FileNotFoundError:
    print("⚠️ Dataset not found. Generating synthetic data for pipeline validation...")
    np.random.seed(42)
    sentiments = ['positive', 'negative', 'neutral']
    df = pd.DataFrame({
        'text': ["I am so happy today!" if i%3==0 else "This is terrible." if i%3==1 else "Just a normal day." for i in range(3000)],
        'selected_text': ["so happy" if i%3==0 else "terrible" if i%3==1 else "normal day" for i in range(3000)],
        'sentiment': [sentiments[i%3] for i in range(3000)]
    })

print("\n--- Data Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

# GRAPH 1: Sentiment Distribution (Neon Bar Chart)
dist_counts = df['sentiment'].value_counts().reset_index()
dist_counts.columns = ['Sentiment', 'Count']
fig1 = px.bar(dist_counts, x='Sentiment', y='Count', 
              title="🎯 Target Distribution (Sentiment Classes)",
              color='Count', color_continuous_scale='Picnic', template="plotly_dark")
fig1.update_layout(title_font_size=20, font_color="#00ffcc")
fig1.show()

📊 STEP 2: DATA INGESTION & EXPLORATORY DATA ANALYSIS (EDA)
✅ Full Dataset Loaded Successfully.

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27481 entries, 0 to 27480
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   textID         27481 non-null  object
 1   text           27480 non-null  object
 2   selected_text  27480 non-null  object
 3   sentiment      27481 non-null  object
dtypes: object(4)
memory usage: 858.9+ KB

--- Missing Values ---
textID           0
text             1
selected_text    1
sentiment        0
dtype: int64


In [3]:
# --- STEP 3: SELECT SUITABLE COLUMNS ---
print("\n🎯 STEP 3: FEATURE SELECTION")
df = df[['text', 'sentiment', 'selected_text']].copy()
print("Selected Columns for Extraction: ['text', 'sentiment', 'selected_text']")

# --- STEP 4: CONVERT CATEGORICAL TO NUMERIC ---
print("\n🔢 STEP 4: CATEGORICAL TRANSFORMATION")
# Encodings for sentiment context (can be concatenated with text embeddings later)
encoder = LabelEncoder()
df['sentiment_id'] = encoder.fit_transform(df['sentiment'])
print("Mapped sentiments to numeric IDs:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))


🎯 STEP 3: FEATURE SELECTION
Selected Columns for Extraction: ['text', 'sentiment', 'selected_text']

🔢 STEP 4: CATEGORICAL TRANSFORMATION
Mapped sentiments to numeric IDs: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [4]:
# --- STEP 5: DATA MANIPULATION & CLEANING ---
print("\n🧹 STEP 5: DATA CLEANSING")

# Drop nulls
initial_shape = df.shape
df.dropna(inplace=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = text.strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df['clean_selected'] = df['selected_text'].apply(clean_text)

print(f"Dropped {initial_shape[0] - df.shape[0]} null rows. Text cleaned (lowercased, URLs removed).")


🧹 STEP 5: DATA CLEANSING
Dropped 1 null rows. Text cleaned (lowercased, URLs removed).


In [5]:
# --- STEP 6: FEATURE ENGINEERING (NLP Meta-features) ---
print("\n⚙️ STEP 6: FEATURE ENGINEERING")

df['text_word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
df['selected_word_count'] = df['clean_selected'].apply(lambda x: len(x.split()))
df['extraction_ratio'] = df['selected_word_count'] / (df['text_word_count'] + 1e-5)

# GRAPH 2: Word Count Distribution (Neon Histogram)
fig2 = px.histogram(df, x="text_word_count", nbins=40, 
                    title="📈 Original Tweet Word Count Distribution",
                    color_discrete_sequence=['#ff007f'], template="plotly_dark")
fig2.update_layout(title_font_size=20)
fig2.show()

# GRAPH 3: Extraction Ratio by Sentiment (Neon Box Plot)
fig3 = px.box(df, x="sentiment", y="extraction_ratio", color="sentiment",
              title="📦 Substring Extraction Ratio by Sentiment",
              template="plotly_dark")
fig3.update_layout(showlegend=False, title_font_size=20)
fig3.show()


⚙️ STEP 6: FEATURE ENGINEERING


In [6]:
# --- STEP 7: ENCODING & SEQUENCING (Tokenization & Target Masking) ---
print("\n🧠 STEP 7: FAST VECTORIZATION & TARGET SEQUENCE GENERATION")
MAX_VOCAB = 25000
MAX_LEN = 35 # Tweets are short

# Tokenizer fitting
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

# 1. Vectorize inputs
X_padded = pad_sequences(tokenizer.texts_to_sequences(df['clean_text']), maxlen=MAX_LEN, padding='post', truncating='post')

# 2. Vectorize targets (to match token IDs)
Y_seq = tokenizer.texts_to_sequences(df['clean_selected'])

# 3. NO FOR-LOOPS: High-speed vectorized masking using numpy
# Generates a binary mask: 1 if the input token exists in the target sequence, else 0
y_padded = np.array([np.isin(x, y).astype(int) for x, y in zip(X_padded, Y_seq)])

# 4. Reshape for TimeDistributed Dense Layer -> (batch, steps, 1)
y_padded = np.expand_dims(y_padded, axis=-1)

print(f"Vocabulary Size: {len(tokenizer.word_index)}")
print(f"X (Input) Tensor Shape: {X_padded.shape}")
print(f"Y (Target Mask) Tensor Shape: {y_padded.shape}")


🧠 STEP 7: FAST VECTORIZATION & TARGET SEQUENCE GENERATION
Vocabulary Size: 25331
X (Input) Tensor Shape: (27480, 35)
Y (Target Mask) Tensor Shape: (27480, 35, 1)


In [7]:
# --- STEP 8: SPLIT X AND Y (Strict Isolation) ---
print("\n✂️ STEP 8: TRAIN/VALIDATION SPLIT")

X_train, X_val, y_train, y_val = train_test_split(
    X_padded, y_padded, 
    test_size=0.15, 
    random_state=42
)

print(f"Training Features: {X_train.shape} | Training Targets: {y_train.shape}")
print(f"Validation Features: {X_val.shape} | Validation Targets: {y_val.shape}")


✂️ STEP 8: TRAIN/VALIDATION SPLIT
Training Features: (23358, 35) | Training Targets: (23358, 35, 1)
Validation Features: (4122, 35) | Validation Targets: (4122, 35, 1)


In [8]:
# --- STEP 9: TRAIN & PREDICT (Model Architecture) ---
print("\n🤖 STEP 9: NEURAL NETWORK TRAINING (SEQUENCE TAGGING)")

# Architecture: Bi-LSTM with TimeDistributed Dense for Token-level prediction
model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN),
    SpatialDropout1D(0.3), # Strict overfitting prevention
    Bidirectional(LSTM(64, return_sequences=True)), # return_sequences=True is critical for sequence tagging
    Dropout(0.3),
    TimeDistributed(Dense(32, activation='relu')),
    TimeDistributed(Dense(1, activation='sigmoid')) # Binary classification per word
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Callbacks to maximize score without endless epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
lr_reduction = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=0.0001, verbose=1)

print("Commencing Model Fitting...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12, 
    batch_size=128,
    callbacks=[early_stop, lr_reduction],
    verbose=1
)


🤖 STEP 9: NEURAL NETWORK TRAINING (SEQUENCE TAGGING)


2026-03-16 17:43:14.239236: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Commencing Model Fitting...
Epoch 1/12
183/183 ━━━━━━━━━━━━━━━━━━━━ 32s 127ms/step - accuracy: 0.8072 - loss: 0.3065 - val_accuracy: 0.8432 - val_loss: 0.2534 - learning_rate: 0.0010
Epoch 2/12
183/183 ━━━━━━━━━━━━━━━━━━━━ 22s 122ms/step - accuracy: 0.8638 - loss: 0.2386 - val_accuracy: 0.8582 - val_loss: 0.2438 - learning_rate: 0.0010
Epoch 3/12
183/183 ━━━━━━━━━━━━━━━━━━━━ 22s 119ms/step - accuracy: 0.8996 - loss: 0.2016 - val_accuracy: 0.8637 - val_loss: 0.2515 - learning_rate: 0.0010
Epoch 4/12
183/183 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.9238 - loss: 0.1618
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
183/183 ━━━━━━━━━━━━━━━━━━━━ 21s 117ms/step - accuracy: 0.9238 - loss: 0.1618 - val_accuracy: 0.8599 - val_loss: 0.2893 - learning_rate: 0.0010
Epoch 5/12
183/183 ━━━━━━━━━━━━━━━━━━━━ 22s 122ms/step - accuracy: 0.9449 - loss: 0.1253 - val_accuracy: 0.8588 - val_loss: 0.3089 - learning_rate: 5.0000e-04
Epoch 5: early stopping
Restoring model 

In [9]:
# --- STEP 10: EVALUATE ACCURACY & PERFORMANCE ---
print("\n🏆 STEP 10: MODEL EVALUATION & METRICS")

loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Final Validation Sequence Accuracy: {accuracy:.2%}")

# GRAPH 4: Training History (Neon Line Chart)
hist_df = pd.DataFrame(history.history)
hist_df['epoch'] = hist_df.index + 1

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=hist_df['epoch'], y=hist_df['accuracy'], name='Train Acc', line=dict(color='#00ffcc', width=3)))
fig4.add_trace(go.Scatter(x=hist_df['epoch'], y=hist_df['val_accuracy'], name='Val Acc', line=dict(color='#ff007f', width=3)))
fig4.update_layout(title='📈 Token Classification Accuracy Trajectory', 
                   xaxis_title='Epoch', yaxis_title='Binary Accuracy per Token',
                   template='plotly_dark', title_font_size=20)
fig4.show()

print("\n" + "="*70)
print("✅ 10-STEP PIPELINE COMPLETE. MLOps ENGINE IS READY FOR DEPLOYMENT.")
print("="*70)


🏆 STEP 10: MODEL EVALUATION & METRICS
Final Validation Sequence Accuracy: 85.82%



✅ 10-STEP PIPELINE COMPLETE. MLOps ENGINE IS READY FOR DEPLOYMENT.


In [10]:
# =========================================================================================
# --- POST-PIPELINE: BULLETPROOF EXPORT & SUBMISSION ---
# =========================================================================================
print("\n📦 POST-PIPELINE: SECURING ENGINE & GENERATING SUBMISSION")
import pickle
import numpy as np

# 1. Save Brain & Dictionary
model.save('tweet_bilstm_engine.keras')
with open('tokenizer.pkl', 'wb') as f: 
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

try:
    # 2. Load Kaggle Data Safely
    test_df = pd.read_csv('/kaggle/input/competitions/tweet-sentiment-extraction/test.csv')
    sub_df = pd.read_csv('/kaggle/input/competitions/tweet-sentiment-extraction/sample_submission.csv')
    
    # Fill any potential NaNs in test data to prevent crashes
    test_df['text'] = test_df['text'].fillna('')
    
    # 3. Vectorize & Predict
    X_test_padded = pad_sequences(
        tokenizer.texts_to_sequences(test_df['text'].str.lower()), 
        maxlen=MAX_LEN, padding='post', truncating='post'
    )
    
    # Predict and create a binary mask (0 or 1)
    preds = model.predict(X_test_padded, verbose=0)
    preds_mask = (preds.squeeze(-1) > 0.5).astype(int)
    
    # 4. Extract Text
    masked_seqs = X_test_padded * preds_mask
    extracted = tokenizer.sequences_to_texts(masked_seqs)
    
    # 5. Clean and Fallback (Crucial for preventing Kaggle's "Empty Value" error)
    final_texts = []
    for i, text in enumerate(extracted):
        clean_t = str(text).strip()
        # If the extraction is empty or just spaces, strictly fallback to the original tweet
        if len(clean_t) == 0:
            final_texts.append(str(test_df['text'].iloc[i]))
        else:
            final_texts.append(clean_t)
            
    # 6. Save to Submission strictly matching Kaggle's format
    sub_df['selected_text'] = final_texts
    sub_df.to_csv('submission.csv', index=False)
    
    print("✅ SUCCESS: 'submission.csv' generated flawlessly with strictly validated strings.")

except Exception as e:
    print(f"⚠️ Warning during inference: {e}")
    print("Submitting sample_submission.csv to bypass Scoring Error...")
    # Ultimate Fail-Safe: If anything fails during hidden test scoring, submit the exact template
    # This guarantees we get a score (even if it's low) instead of a fatal error!
    fallback_sub = pd.read_csv('../input/tweet-sentiment-extraction/sample_submission.csv')
    fallback_sub.to_csv('submission.csv', index=False)


📦 POST-PIPELINE: SECURING ENGINE & GENERATING SUBMISSION
✅ SUCCESS: 'submission.csv' generated flawlessly with strictly validated strings.


# 🌐 SYSTEM DEPLOYMENT: FROM NOTEBOOK TO PRODUCTION
> **"True intelligence is not just classification; it is the precise extraction of meaning."**

The 10-Step Neural Pipeline for **Information Extraction** has successfully executed. We have transitioned from basic text classification to advanced **Token-Level Sequence Tagging**, utilizing a robust **Bidirectional LSTM** to autonomously extract the exact substrings that carry emotional and semantic weight.

The neural extraction brain (`.keras`) and its vectorization dictionary (`tokenizer.pkl`) have been exported and are now serving real-time, vectorized inference in a cloud environment.

### 🚀 Test the Live Extraction Engine
You can test this AI architecture interactively via the Streamlit web application deployed on Hugging Face Spaces. Enter any text, and watch the engine extract the core sentiment phrase instantly:

👉 **[Launch the Neural Extraction Engine (Hugging Face)](https://huggingface.co/spaces/Ironside35/tweet-sentiment-extractor)**

---
**Architecture:** Sequence Tagging via Deep Bi-LSTM (TimeDistributed)  
**Frameworks:** TensorFlow, Keras, Plotly, Streamlit  
*Engineered for Precision Information Extraction | March 2026*